In [ ]:
pip install boto3

In [ ]:
pip install pyarrow

In [1]:
import os
import datetime
import shutil
import pandas as pd
import json
import subprocess
import boto3
import zipfile
import sys
# Add the parent directory to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from pathlib import Path
from datetime import datetime


In [2]:

# Add the parent directory to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))



In [4]:
# Define base path
base_path = "s3://oedi-data-lake/nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/2024/resstock_tmy3_release_2/model_and_schedule_files/building_energy_models/upgrade=0"

# List of building IDs
#building_ids = [539175,533874,506556,494283,464636,459711,437325,431540,420644,406755,405914,404945,357101,288490,281301,229321,219255,218212,197295,180093,170879,129064,115010,71175,63786,52401,46648,23156,12618,11933,10127]
building_ids = [262549,505105,140964, 474519,335282, 205031,404842,275941,515274,332812]
folder_path = "resstock_2024"
weather_file = "Ithaca_2024.epw"

current_directory = Path(os.getcwd())
base_files = os.path.join(current_directory,'base_files')

# Check if the folder exists
if not os.path.exists(folder_path):
    os.makedirs(folder_path, exist_ok=True) 
    print(f"Folder created at: {folder_path}")
else:
    print("Folder already exists.")

    

Folder created at: resstock_2024


In [9]:
import os, json, shutil, subprocess, zipfile
from pathlib import Path

# ... your existing setup for:
# base_files, base_path, folder_path, building_ids, weather_file

for bid in building_ids:
    filename = f"bldg{bid:07d}-up00.zip"
    str_bid = str(bid)

    # make folder for building ID
    bdir = os.path.join(folder_path, str_bid)
    os.makedirs(bdir, exist_ok=True)

    # make models folder
    models = os.path.join(bdir, 'models')
    os.makedirs(models, exist_ok=True)
    print("models dir:", models)

    # MEASURES (delete-and-recopy if you uncomment later)
    new_measure_path = os.path.join(bdir, 'measures')
    if os.path.exists(new_measure_path):
        try:
            shutil.rmtree(new_measure_path)
            print(f"Deleted existing folder at: {new_measure_path}")
        except Exception as e:
            print(f"Error deleting folder: {e}")
            raise
    # shutil.copytree(measure_path, new_measure_path)

    # WEATHER
    weather_original = os.path.join(base_files, 'weather', weather_file)
    weather_new = os.path.join(bdir, 'weather')
    os.makedirs(weather_new, exist_ok=True)
    shutil.copy(weather_original, weather_new)

    # WORKFLOW.OSW
    osw_path = Path(os.path.join(base_files, 'workflow_resstock.osw'))
    new_osw_path = os.path.join(bdir, 'workflow_resstock.osw')
    shutil.copy(osw_path, new_osw_path)

    with open(new_osw_path, 'r') as f:
        data = json.load(f)
    data['seed_file'] = "in.osm"
    data['weather_file'] = weather_file
    with open(new_osw_path, 'w') as f:
        json.dump(data, f, indent=2)

    # OSM ZIP FROM S3
    s3_path = f"{base_path}/{filename}"

    print(s3_path)
    print(f"Downloading {filename} to {models}...")
    r = subprocess.run(
        ["aws", "s3", "cp", s3_path, models, "--no-sign-request"],
        capture_output=True, text=True
    )

    if r.returncode != 0:
        print("Download failed ❌")
        print("STDERR:", r.stderr)
        continue

    # AWS CLI prints progress to STDERR; not having STDOUT is normal
    if not os.path.exists(dest_zip) or os.path.getsize(dest_zip) == 0:
        print("Download did not produce a file at:", dest_zip)
        print("STDERR:", r.stderr)
        continue

    print(f"Unzipping {dest_zip} → {models}")
    with zipfile.ZipFile(dest_zip, 'r') as zf:
        zf.extractall(models)

print('done')


models dir: resstock_2024/262549/models
s3://oedi-data-lake/nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/2024/resstock_tmy3_release_2/model_and_schedule_files/building_energy_models/upgrade=0/bldg0262549-up00.zip
Download did not produce a file at: resstock_2024/332812/models/bldg0332812-up00.zip
STDERR: 
models dir: resstock_2024/505105/models
s3://oedi-data-lake/nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/2024/resstock_tmy3_release_2/model_and_schedule_files/building_energy_models/upgrade=0/bldg0505105-up00.zip
Download did not produce a file at: resstock_2024/332812/models/bldg0332812-up00.zip
STDERR: 
models dir: resstock_2024/140964/models
s3://oedi-data-lake/nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/2024/resstock_tmy3_release_2/model_and_schedule_files/building_energy_models/upgrade=0/bldg0140964-up00.zip
Download did not produce a file at: resstock_2024/332812/models/bldg0332812-up00.zip
STDERR: 
models dir